# Random Networks

This notebook contains demos for:
- Erdős-Rényi Model
- "Copy and Delete" Model
- Geometric Model
- Watts-Strogatz Small-World Model
- Barabási-Albert Model vs Erdős-Rényi Model comparison
- ERGM-like features for Zachary's Karate Club network

## Setup

In [ ]:
# If needed, install dependencies (uncomment the next line)
# !pip install networkx matplotlib numpy

import math
import random
from collections import Counter
from typing import List, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx


# Giant component subgraph and ASP on giant component
def giant_component_subgraph(G: nx.Graph) -> nx.Graph:
    if G.number_of_nodes() == 0:
        return G
    comp = max(nx.connected_components(G), key=len)
    return G.subgraph(comp).copy()


def average_shortest_path_on_giant(G: nx.Graph) -> float:
    if G.number_of_nodes() == 0:
        return 0.0
    H = giant_component_subgraph(G)
    if H.number_of_nodes() <= 1:
        return 0.0
    return nx.average_shortest_path_length(H)


def plot_graph(G: nx.Graph, pos: Optional[Dict] = None, title: str = "Graph", **kwargs):
    plt.figure()
    if pos is None:
        pos = nx.spring_layout(G, seed=7)
    nx.draw(G, pos, with_labels=True, node_size=300, font_size=8, **kwargs)
    plt.title(title)
    plt.axis('off')
    plt.show()



## 1) Erdős–Rényi Model

**Tasks**
1. Generate and visualize a small ER network using the G(n,p) formulation. Annotate the giant component fraction.
2. Sweep $p$ to observe the phase transition (giant component emergence at about $p\approx 1/n$).
3. Compare empirical degree distribution to *Binomial* and *Poisson* predictions.


In [ ]:
# 1.1 Visualize a small G(n,p) and the largest component
n, p, seed = 20, 0.05, 42
G = nx.erdos_renyi_graph(n, p, seed=seed)

# Find the largest connected component
lcc = max(nx.connected_components(G), key=len)
gc_frac = len(lcc) / n if n > 0 else 0

# Highlight the giant component in orange
node_color = ['tab:cyan' if v in lcc else 'lightgray' for v in G.nodes()]
edge_color = ['tab:cyan' if (u in lcc and v in lcc) else 'lightgray' for u, v in G.edges()]

plot_graph(
    G,
    title=f"Erdős–Rényi G(n={n}, p={p}) | Giant fraction={gc_frac:.2f}",
    node_color=node_color,
    edge_color=edge_color
)


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

n, seed, layout_seed = 100, 42, 7
pos = nx.spring_layout(nx.complete_graph(n), seed=layout_seed)

fig, ax = plt.subplots(figsize=(6, 6))
ax.axis('off')
plt.close(fig)  # avoid duplicate empty output in notebooks

def redraw(p):
    ax.clear()
    ax.axis('off')
    G = nx.erdos_renyi_graph(n, p, seed=seed)
    lcc = max(nx.connected_components(G), key=len) if G.number_of_nodes() else set()
    gc_frac = len(lcc)/n if n>0 else 0
    node_color = ['tab:orange' if v in lcc else 'lightgray' for v in G.nodes()]
    edge_color = ['tab:orange' if (u in lcc and v in lcc) else 'lightgray' for u, v in G.edges()]
    nx.draw(G, pos, with_labels=False, node_size=60, width=1,
            node_color=node_color, edge_color=edge_color, ax=ax)
    ax.set_title(f"G(n={n}, p={p:.3f}) | Giant fraction={gc_frac:.2f}")
    display(fig)

slider = widgets.FloatSlider(value=0.00, min=0.0, max=0.1, step=0.001, description='p')
out = widgets.interactive_output(redraw, {'p': slider})
display(slider, out)

In [ ]:
# 1.3 Binomial and Poisson predictions
n = 100
p_values = [0.0025, 0.005, 0.0075, 0.01, 0.025, 0.05]
seed = 1

for p in p_values:
    G = nx.erdos_renyi_graph(n, p, seed=seed)
    gc = max(nx.connected_components(G), key=len)
    gc_frac = len(gc) / n
    degrees = [d for _, d in G.degree()]
    plt.figure()
    bins = np.arange(min(degrees), max(degrees) + 2) - 0.5 if degrees else np.arange(0, 1.5) - 0.5
    plt.hist(degrees, bins=bins, density=True)  # empirical
    # Theoretical curves
    k_vals = np.arange(0, n)
    binom = [math.comb(n - 1, k) * (p ** k) * ((1 - p) ** (n - 1 - k)) for k in k_vals]
    lam = p * (n - 1)
    pois = [math.exp(-lam) * lam ** k / math.factorial(k) for k in k_vals]
    plt.plot(k_vals, binom, label="Binomial(n-1,p)")
    plt.plot(k_vals, pois, label="Poisson(λ=(n-1)p)")
    plt.legend()
    plt.xlabel("Degree k")
    plt.ylabel("Probability")
    plt.title(f"G(n={n}, p={p}) | Giant fraction={gc_frac:.2f}")
    plt.show()


## 2) "Copy and Delete" Model

**Mechanism (simple variant):**
- Pick random node $u$.
- For each neighbor $v$ of $u$, with probability $p_{copy}$, connect a random node $w$ to $v$.
- With probability $p_{delete}$, delete a random existing edge.
- Periodically add a new isolated node.

**Task**
- Track *degree distribution* evolution at multiple time steps.

In [ ]:
def copy_delete_model(n0: int,
                      steps: int,
                      p_copy: float,
                      p_delete: float,
                      add_new_node_every: int,
                      seed: int,
                      p_seed: float = 0.8,         # seed edge for new nodes
                      p_copy_to_new: float = 0.25  # sometimes copy to the newest node
                      ) -> nx.Graph:
    rng = random.Random(seed)
    G = nx.erdos_renyi_graph(n0, 0.3, seed=seed)
    newest = None

    for t in range(1, steps + 1):
        if G.number_of_nodes() == 0:
            G.add_node(0)
        u = rng.choice(list(G.nodes()))
        if G.degree(u) > 0:
            neighbors = list(G.neighbors(u))

            # pick w (sometimes force it to be the newest node, if any)
            if newest is not None and rng.random() < p_copy_to_new:
                w = newest
            else:
                w = rng.choice(list(G.nodes()))

            for v in neighbors:
                if w != v and rng.random() < p_copy:
                    G.add_edge(w, v)

        if rng.random() < p_delete and G.number_of_edges() > 0:
            e = rng.choice(list(G.edges()))
            G.remove_edge(*e)

        if (t % add_new_node_every) == 0:
            new_id = (max(G.nodes()) + 1) if len(G) else 0
            G.add_node(new_id)
            newest = new_id
            # seed an attachment so the node is not isolated forever
            if len(G) > 1 and rng.random() < p_seed:
                # connect to a random existing node
                u0 = rng.choice([x for x in G.nodes() if x != new_id])
                G.add_edge(new_id, u0)
        else:
            newest = newest  # unchanged

    return G



def track_degree_evolution(
        checkpoints: List[int],
        n0: int,
        p_copy: float,
        p_delete: float,
        add_new_node_every: int,
        seed: int
):
    max_steps = max(checkpoints)
    deg_histories = {}
    for t in range(1, max_steps + 1):
        G = copy_delete_model(
            n0=n0,
            steps=t,
            p_copy=p_copy,
            p_delete=p_delete,
            add_new_node_every=add_new_node_every,
            seed=seed
        )
        if t in checkpoints:
            deg = [d for _, d in G.degree()]
            hist = Counter(deg)
            deg_histories[t] = (hist, G)
    for t in checkpoints:
        hist, Gt = deg_histories[t]
        xs = sorted(hist.keys())
        ys = [hist[x] / sum(hist.values()) for x in xs]
        plt.figure()
        plt.bar(xs, ys)
        plt.xlabel("Degree k")
        plt.ylabel("Frequency")
        plt.title(f"Copy–Delete Degree Distribution @ step {t}")
        plt.show()
    return deg_histories


# --- Define model ---
n0 = 10
steps = 100
seed = 7
checkpoints = [25, 50, 75, 100]

# Clear giant-component growth (balanced)
p_copy=0.75
p_delete=0.05
add_new_node_every=2

# Strong hub/heterogeneity (copy-dominant)
# p_copy=0.9
# p_delete=0.02
# add_new_node_every=5

# High churn (more visible edge turnover)
# p_copy=0.7
# p_delete=0.12
# add_new_node_every=2

# --- Run the model ---
G_cd = copy_delete_model(
    n0=n0,
    steps=steps,
    p_copy=p_copy,
    p_delete=p_delete,
    add_new_node_every=add_new_node_every,
    seed=seed
)

pos = nx.kamada_kawai_layout(G_cd)
plot_graph(G_cd, pos=pos, title="\"Copy and Delete\" Model (final state)")

# --- Track evolution at checkpoints ---
_ = track_degree_evolution(
    checkpoints=checkpoints,
    n0=n0,
    p_copy=p_copy,
    p_delete=p_delete,
    add_new_node_every=add_new_node_every,
    seed=seed
)


## 4) Geometric Random Graph

**Tasks**
- Place nodes uniformly at random in $[0,1]^2$.
- Connect nodes within radius $r$.
- Vary $r$ to study connectivity thresholds.
- Visualize and report giant component fraction.

In [ ]:
n = 120
r_values = [0.050, 0.075, 0.100, 0.125, 0.150]
seed = 3

for r in r_values:
    G = nx.random_geometric_graph(n, r, seed=seed)
    pos = nx.get_node_attributes(G, "pos")
    lcc = max(nx.connected_components(G), key=len)
    gc_frac = len(lcc) / n
    node_color = ['tab:cyan' if v in lcc else 'lightgray' for v in G.nodes()]
    edge_color = ['tab:cyan' if (u in lcc and v in lcc) else 'lightgray' for u, v in G.edges()]
    plot_graph(G, pos=pos, title=f"Geometric RG (n={n}, r={r}) | Giant fraction={gc_frac:.2f}",node_color=node_color,edge_color=edge_color)


## 5) Watts–Strogatz Small‑World Model

**Tasks**
- Start from ring lattice (degree $k$), rewire each edge with probability $\beta$.
- Measure *average clustering*, *transitivity*, and *average shortest path*.
- Relate to small‑world phenomenon: high clustering + short paths at small $\beta$.

In [ ]:
n, k, betas, seed = 80, 6, [0.0, 0.05, 0.2, 0.8], 11

for beta in betas:
    G = nx.watts_strogatz_graph(n, k, beta, seed=seed)
    H = giant_component_subgraph(G)
    avg_clust = nx.average_clustering(G)
    trans = nx.transitivity(G)
    asp = nx.average_shortest_path_length(H) if H.number_of_nodes() > 1 else 0.0
    pos = nx.circular_layout(G)
    plot_graph(G, pos=pos, title=(
        f"Watts–Strogatz (n={n}, k={k}, β={beta}) | "
        f"C_local={avg_clust:.2f}, trans={trans:.2f}, ASP(giant)={asp:.2f}"
    ))


## 6) Barabási–Albert Model vs Erdős–Rényi Model comparison (degree distributions)

**Task**

Compare degree heterogeneity in BA vs ER.

*Expectation:* BA shows a heavier tail than ER.

In [ ]:
n, m, p, seed = 200, 2, 0.02, 5
G_ba = nx.barabasi_albert_graph(n, m, seed=seed)
G_er = nx.erdos_renyi_graph(n, p, seed=seed)

for name, G in [("BA", G_ba), ("ER", G_er)]:
    degrees = [d for _, d in G.degree()]
    plt.figure()
    bins = np.arange(min(degrees), max(degrees) + 2) - 0.5
    plt.hist(degrees, bins=bins, density=True)
    plt.xlabel("Degree k")
    plt.ylabel("Frequency")
    plt.title(f"{name} degree distribution (n={n})")
    plt.show()


## 3) ERGM-like features (pattern exploration)

**Task**

Use a small real-world network (Zachary's Karate Club) to inspect features that ERGMs aim to model:
- Edges and degree distribution
- Triangles, average clustering, transitivity
- Degree assortativity
- Components and average shortest path on the giant component

In [ ]:
Gk = nx.karate_club_graph()
triangles = sum(nx.triangles(Gk).values()) // 3
avg_clust = nx.average_clustering(Gk)
transitivity = nx.transitivity(Gk)
assort = nx.degree_assortativity_coefficient(Gk)
comps = list(nx.connected_components(Gk))
H = giant_component_subgraph(Gk)
asp = nx.average_shortest_path_length(H)

print("Karate Club (ERGM-style features)")
print(f" - n = {Gk.number_of_nodes()}, m = {Gk.number_of_edges()}")
print(f" - triangles = {triangles}")
print(f" - average_clustering = {avg_clust:.3f}")
print(f" - transitivity = {transitivity:.3f}")
print(f" - degree_assortativity = {assort:.3f}")
print(f" - components = {len(comps)}, giant_size = {H.number_of_nodes()}")
print(f" - avg_shortest_path_length (giant) = {asp:.3f}")

plot_graph(Gk, title="Karate Club Graph")
